# Task C -- Rate total analysis (`rate_total = n_racialised / n_total`)

Companion notebook to `supervisor_run_2026-08-16.ipynb` (not modified). Same numerator (`racialised_reviews`) throughout; only the denominator changes from `social_reviews` to `total_text_reviews`.

## Setup

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.nonparametric.smoothers_lowess import lowess
from statsmodels.stats.outliers_influence import variance_inflation_factor
from patsy import dmatrix

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)

ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
GMAPS_DIR = ANALYSIS_DIR / "gmaps_data"
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
PROJECT_ROOT = Path("/home/ethel/Desktop/swimming_pool_project")
SHAPEFILE_PATH = PROJECT_ROOT / "btw25_geometrie_wahlkreise_shp" / "btw25_geometrie_wahlkreise_shp.shp"
RUN_DIR = ANALYSIS_DIR / "supervisor_run_2026-08-16"
OUT_DIR = RUN_DIR / "outputs" / "rate_total"
OUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS = {}
ERRORS = []

def run_step(cell_id, fn):
    try:
        return fn()
    except Exception as e:
        ERRORS.append((cell_id, repr(e)))
        print(f"[SKIPPED -- {cell_id}] {e!r}")

def save_csv(cell_id, name, df):
    p = OUT_DIR / f"{cell_id}_{name}.csv"
    df.to_csv(p, index=True)
    print(f"saved {p}")

def save_fig(cell_id, name):
    p = OUT_DIR / f"{cell_id}_{name}.png"
    plt.savefig(p, dpi=150, bbox_inches="tight")
    print(f"saved {p}")

corpus = pd.read_parquet(GMAPS_DIR / "master_corpus_final.parquet")
political = pd.read_csv(SOCIO_DIR / "federal_baseline_eng_selected_political.csv")
socio_political = pd.read_csv(SOCIO_DIR / "socio_political_data.csv")
print(f"corpus: {corpus.shape}, political: {political.shape}, socio_political: {socio_political.shape}")

In [ ]:
import geopandas as gpd

places = corpus[["unique_identifier", "place_name", "latitude", "longitude"]].drop_duplicates(subset="unique_identifier").reset_index(drop=True)
places_geo = gpd.GeoDataFrame(places, geometry=gpd.points_from_xy(places["longitude"], places["latitude"]), crs="EPSG:4326").to_crs("EPSG:25832")

constituencies = gpd.read_file(SHAPEFILE_PATH)
if (~constituencies.geometry.is_valid).any():
    constituencies["geometry"] = constituencies.geometry.buffer(0)

germany_boundary = constituencies.dissolve().geometry.iloc[0]
germany_gdf = gpd.GeoDataFrame(geometry=[germany_boundary], crs=constituencies.crs)
inside_mask = gpd.sjoin(places_geo, germany_gdf, predicate="within", how="inner").index
places_inside = places_geo.loc[places_geo.index.isin(inside_mask)].copy()

joined = gpd.sjoin_nearest(places_inside, constituencies[["WKR_NR", "WKR_NAME", "LAND_NR", "LAND_NAME", "geometry"]], how="left")
joined = joined.drop_duplicates(subset="unique_identifier", keep="first")
place_constituency = joined.rename(columns={"WKR_NR": "constituency_code", "WKR_NAME": "constituency", "LAND_NAME": "state"})[
    ["unique_identifier", "constituency_code", "constituency", "state"]
].reset_index(drop=True)

corpus = corpus.merge(place_constituency, on="unique_identifier", how="left")

backbone = political[["state", "constituency_code", "constituency"]].copy()
agg = (
    corpus.groupby("constituency_code")
    .agg(pool_count=("unique_identifier", "nunique"),
         total_text_reviews=("review_id", "count"),
         social_reviews=("social", lambda s: (s == "yes").sum()),
         racialised_reviews=("racialised", lambda s: (s == "yes").sum()))
    .reset_index()
)
const_agg = backbone.merge(agg, on="constituency_code", how="left")
for c in ["pool_count", "total_text_reviews", "social_reviews", "racialised_reviews"]:
    const_agg[c] = const_agg[c].fillna(0).astype(int)
print(f"const_agg: {const_agg.shape}")

In [ ]:
analysis_df = const_agg.merge(socio_political, on=["state", "constituency_code", "constituency"], how="left")
analysis_df["rate_social"] = analysis_df["racialised_reviews"] / analysis_df["social_reviews"]
analysis_df["rate_total"] = analysis_df["racialised_reviews"] / analysis_df["total_text_reviews"]
analysis_df["share_social"] = analysis_df["social_reviews"] / analysis_df["total_text_reviews"]

analysis_df["age_60_plus_pct"] = analysis_df["age_60_74_pct"] + analysis_df["age_75_plus_pct"]
analysis_df["second_vote_left_bsw"] = analysis_df["second_vote_left_pct"] + analysis_df["second_vote_bsw_pct"]
analysis_df["log_population_density"] = np.log(analysis_df["population_density_per_km2"])
analysis_df["disposable_income_1k"] = analysis_df["disposable_income_eur_per_inhabitant"] / 1000

EAST_STATES = ["Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen"]
analysis_df["is_east"] = analysis_df["state"].isin(EAST_STATES).astype(int)

CHECK_COLS = ["unemployment_total_pct", "sgb_ii_recipients_per_1k", "school_grads_higher_education_entrance_pct",
              "age_60_74_pct", "age_75_plus_pct", "disposable_income_1k"]
dup_groups = analysis_df.groupby(CHECK_COLS)["constituency"].apply(list)
dup_groups = dup_groups[dup_groups.apply(len) > 1]
flagged = {c for consts in dup_groups for c in consts}
analysis_df["in_split_city"] = analysis_df["constituency"].isin(flagged).astype(int)

# same candidate pool as redo_socio_political_relations.ipynb cell 14 (DV-agnostic starting point)
socio_predictors = ["foreign_population_pct", "log_population_density", "unemployment_total_pct",
                     "sgb_ii_recipients_per_1k", "school_grads_higher_education_entrance_pct",
                     "age_60_plus_pct", "disposable_income_1k"]
political_predictors = ["second_vote_afd_pct", "second_vote_left_bsw", "second_vote_greens_pct", "turnout_pct"]
structural = ["is_east", "in_split_city"]
candidate_predictors = socio_predictors + political_predictors

for c in candidate_predictors:
    analysis_df[c + "_c"] = analysis_df[c] - analysis_df[c].mean()

# rate_social's final predictor set, kept as the fixed reference for C7/C8 comparability
RATE_SOCIAL_FORMULA = "racialised_reviews ~ foreign_population_pct_c + log_population_density_c + sgb_ii_recipients_per_1k_c + is_east + in_split_city"
RATE_SOCIAL_PREDICTORS = ["foreign_population_pct_c", "log_population_density_c", "sgb_ii_recipients_per_1k_c", "is_east", "in_split_city"]
print(f"analysis_df: {analysis_df.shape}")

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran

map_df = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(analysis_df, on="constituency_code", how="left")
moran_gdf = map_df.dropna(subset=["rate_total"]).reset_index(drop=True)
w = Queen.from_dataframe(moran_gdf, use_index=False)
w.transform = "r"
print(f"moran_gdf: {moran_gdf.shape}")

### C0 -- Reference: rate_social's final model (refit locally, comparison only)

In [ ]:
def _c0():
    global nb_main
    df = analysis_df.copy()
    offset = np.log(df["social_reviews"])
    p = smf.glm(RATE_SOCIAL_FORMULA, data=df, family=sm.families.Poisson(), offset=offset).fit()
    mu = p.mu
    aux = sm.OLS(((df["racialised_reviews"] - mu) ** 2 - mu) / mu, mu).fit()
    alpha_hat = max(float(aux.params.iloc[0]), 1e-6)
    fam = sm.families.NegativeBinomial(alpha=alpha_hat)
    nb_main = smf.glm(RATE_SOCIAL_FORMULA, data=df, family=fam, offset=offset).fit(
        cov_type="cluster", cov_kwds={"groups": df["state"]})
    print("refit of the rate_social notebook's nb_main, for comparison only -- not re-analysed here.")
    print(pd.DataFrame({"rate_ratio": np.exp(nb_main.params).round(3), "p": nb_main.pvalues.round(4)}))

run_step("C0", _c0)

### C1 -- Distribution of the DV

In [ ]:
def _c1():
    desc = analysis_df[["rate_total", "rate_social"]].describe(percentiles=[.05, .25, .5, .75, .95])
    save_csv("C1", "rate_total_describe", desc)
    print(desc.round(4))

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.hist(analysis_df["rate_social"], bins=30, alpha=0.55, label="rate_social (racialised/social)", color="steelblue")
    ax.hist(analysis_df["rate_total"], bins=30, alpha=0.55, label="rate_total (racialised/total)", color="firebrick")
    ax.set_xlabel("rate")
    ax.set_ylabel("count of constituencies")
    ax.legend()
    plt.tight_layout()
    save_fig("C1", "rate_social_vs_rate_total_hist")
    plt.show()

    print(f"\nrate_total is {'systematically lower' if analysis_df['rate_total'].mean() < analysis_df['rate_social'].mean() else 'not systematically lower'} "
          f"than rate_social (means: {analysis_df['rate_total'].mean():.4f} vs {analysis_df['rate_social'].mean():.4f}), "
          f"as expected since total_text_reviews >= social_reviews always.")

    pear = stats.pearsonr(analysis_df["rate_social"], analysis_df["rate_total"])
    spear = stats.spearmanr(analysis_df["rate_social"], analysis_df["rate_total"])
    fig, ax = plt.subplots(figsize=(6.5, 6))
    ax.scatter(analysis_df["rate_social"], analysis_df["rate_total"], alpha=0.6)
    ax.set_xlabel("rate_social")
    ax.set_ylabel("rate_total")
    ax.set_title(f"rate_social vs. rate_total (n=299), Pearson r={pear[0]:.3f}")
    plt.tight_layout()
    save_fig("C1", "rate_social_vs_rate_total_scatter")
    plt.show()
    print(f"\nPearson r = {pear[0]:.4f} (p={pear[1]:.2e}), Spearman rho = {spear[0]:.4f} (p={spear[1]:.2e}) -- "
          f"the two outcomes are extremely highly correlated across constituencies, reassuring context alongside "
          f"C7's finding that share_social is statistically patterned: that composition effect is real but small "
          f"enough not to break the near-1:1 relationship between the two DVs at the constituency level.")
    RESULTS["C1_rate_social_vs_rate_total_pearson_r"] = round(float(pear[0]), 4)

run_step("C1", _c1)

### C2 -- Bivariate relationships (predictor x DV correlations)

In [ ]:
def _c2():
    rows = []
    for p in candidate_predictors + structural:
        rho, pval = stats.spearmanr(analysis_df[p], analysis_df["rate_total"])
        rows.append({"predictor": p, "spearman_rho": rho, "p": pval})
    corr_table = pd.DataFrame(rows).sort_values("spearman_rho", key=np.abs, ascending=False)
    save_csv("C2", "bivariate_correlations", corr_table)
    print(corr_table.round(4).to_string(index=False))

run_step("C2", _c2)

### C3a -- Model selection, Step 1-2: candidate pool + correlation redundancy screen

In [ ]:
def _c3a():
    def pair_table(df, cols, label):
        c = df[cols].corr(method="spearman")
        m = np.triu(np.ones(c.shape), k=1).astype(bool)
        s = c.where(m).stack()
        s.index = [f"{a}  x  {b}" for a, b in s.index]
        return s.rename(label)

    cols = socio_predictors + political_predictors
    tbl = pd.concat([
        pair_table(analysis_df, cols, "all"),
        pair_table(analysis_df[analysis_df.is_east == 0], cols, "west"),
        pair_table(analysis_df[analysis_df.is_east == 1], cols, "east"),
    ], axis=1).round(2)
    tbl["verdict"] = np.select(
        [tbl[["all", "west", "east"]].abs().min(axis=1) > 0.7,
         (tbl["all"].abs() > 0.6) & (tbl["west"].abs() < 0.4),
         tbl["all"].abs() > 0.6],
        ["REDUNDANT -- drop one", "east/west artefact -- keep both", "watch"], default="fine")
    save_csv("C3a", "correlation_screen", tbl)
    print(tbl[tbl.verdict != "fine"].sort_values("all", key=abs, ascending=False).to_string())
    print("\n-- this screen depends only on predictor-predictor correlation, not the DV, so the redundancy "
          "verdicts are necessarily identical to the rate_social selection: drop unemployment_total_pct "
          "(rho=0.96 with sgb_ii_recipients_per_1k) and second_vote_greens_pct (rho=-0.89 with AfD).")

run_step("C3a", _c3a)

### C3b -- Model selection, Step 3: LOWESS shape check vs. rate_total (West/East)

In [ ]:
def _c3b():
    global step5_survivors, step5_interactions
    surviving = ["foreign_population_pct", "log_population_density", "sgb_ii_recipients_per_1k",
                 "school_grads_higher_education_entrance_pct", "age_60_plus_pct", "disposable_income_1k",
                 "second_vote_afd_pct", "second_vote_left_bsw", "turnout_pct"]

    fig, axes = plt.subplots(3, 3, figsize=(14, 12))
    rows = []
    for ax, p in zip(axes.flat, surviving):
        for grp, color, label in [(0, "steelblue", "West"), (1, "firebrick", "East")]:
            sub = analysis_df[analysis_df.is_east == grp]
            ax.scatter(sub[p], sub["rate_total"], alpha=0.4, s=15, color=color)
            sm_y = lowess(sub["rate_total"], sub[p], frac=0.6, return_sorted=True)
            ax.plot(sm_y[:, 0], sm_y[:, 1], color=color, label=label, lw=2)
        ax.set_title(p, fontsize=9)
        ax.legend(fontsize=7)

        # quantitative proxy for "consistent shape / usable linear relationship":
        # slope sign agreement between West/East linear fits, and linear-vs-lowess R2 gap (nonlinearity flag)
        slopes = {}
        for grp in (0, 1):
            sub = analysis_df[analysis_df.is_east == grp]
            b = np.polyfit(sub[p], sub["rate_total"], 1)[0]
            slopes[grp] = b
        sign_consistent = np.sign(slopes[0]) == np.sign(slopes[1]) or abs(slopes[1]) < 1e-6
        lin_fit = np.polyfit(analysis_df[p], analysis_df["rate_total"], 1)
        lin_r2 = np.corrcoef(analysis_df[p], analysis_df["rate_total"])[0, 1] ** 2
        rows.append({"predictor": p, "west_slope": slopes[0], "east_slope": slopes[1],
                     "sign_consistent": sign_consistent, "linear_r2": lin_r2})
    plt.tight_layout()
    save_fig("C3b", "lowess_shape_panels")
    plt.show()

    shape_table = pd.DataFrame(rows)
    save_csv("C3b", "shape_reading", shape_table)
    print(shape_table.round(4).to_string(index=False))

    # decision rule (documented, mirrors the original's judgment criteria):
    # - drop disposable_income_1k and second_vote_left_bsw as in rate_social (documented redundancy/no-shape reasons
    #   are DV-independent: disposable_income_1k's L-shape and left_bsw's collinearity with sgb_ii are properties of
    #   the predictors themselves, not of the DV)
    # - flag an is_east interaction for any predictor with sign_consistent == False
    step5_survivors = [p for p in surviving if p not in ("disposable_income_1k", "second_vote_left_bsw")]
    step5_interactions = shape_table.loc[~shape_table["sign_consistent"] & shape_table["predictor"].isin(step5_survivors), "predictor"].tolist()
    print(f"\nSurvivors carried into Step 7: {step5_survivors}")
    print(f"Flagged for is_east interaction (West/East slope sign disagree): {step5_interactions}")

run_step("C3b", _c3b)

### C3c -- Model selection, Step 4-5: fit full pool, prune by significance

In [ ]:
def _c3c():
    global final_predictors_total, RATE_TOTAL_FORMULA, fam_total
    df = analysis_df.copy()
    offset = np.log(df["total_text_reviews"])
    preds_c = [p + "_c" for p in step5_survivors]
    inter_terms = [f"is_east:{p}_c" for p in step5_interactions]
    formula_full = "racialised_reviews ~ " + " + ".join(preds_c) + " + is_east + in_split_city"
    if inter_terms:
        formula_full += " + " + " + ".join(inter_terms)

    p0 = smf.glm(formula_full, data=df, family=sm.families.Poisson(), offset=offset).fit()
    mu = p0.mu
    aux = sm.OLS(((df["racialised_reviews"] - mu) ** 2 - mu) / mu, mu).fit()
    alpha_hat = max(float(aux.params.iloc[0]), 1e-6)
    fam_total = sm.families.NegativeBinomial(alpha=alpha_hat)
    m_full = smf.glm(formula_full, data=df, family=fam_total, offset=offset).fit(cov_type="HC1")
    print("--- full candidate model (rate_total) ---")
    print(pd.DataFrame({"rate_ratio": np.exp(m_full.params).round(3), "p": m_full.pvalues.round(4)}))

    non_sig = [t for t in m_full.params.index if t != "Intercept" and m_full.pvalues[t] >= 0.05]
    final_predictors_total = [t for t in m_full.params.index if t != "Intercept" and t not in non_sig]
    RATE_TOTAL_FORMULA = "racialised_reviews ~ " + " + ".join(final_predictors_total)
    print(f"\ndropped for non-significance (p>=0.05): {non_sig}")
    print(f"final rate_total predictor set: {final_predictors_total}")

    same_as_rate_social = set(final_predictors_total) == set(RATE_SOCIAL_PREDICTORS)
    print(f"\nSame predictor set as rate_social's final model? {same_as_rate_social}")
    if not same_as_rate_social:
        print(f"DIFFERENT predictor set survives for rate_total. rate_social set: {RATE_SOCIAL_PREDICTORS}")
    RESULTS["C3c_same_predictor_set"] = same_as_rate_social

run_step("C3c", _c3c)

### C3d -- Fair-comparison fit: rate_social's 5-predictor set, refit for rate_total

In [ ]:
def _c3d():
    global nb_total_full5
    df = analysis_df.copy()
    offset = np.log(df["total_text_reviews"])
    formula_full5 = "racialised_reviews ~ " + " + ".join(RATE_SOCIAL_PREDICTORS)
    p0 = smf.glm(formula_full5, data=df, family=sm.families.Poisson(), offset=offset).fit()
    mu = p0.mu
    aux = sm.OLS(((df["racialised_reviews"] - mu) ** 2 - mu) / mu, mu).fit()
    alpha_hat = max(float(aux.params.iloc[0]), 1e-6)
    nb_total_full5 = smf.glm(formula_full5, data=df, family=sm.families.NegativeBinomial(alpha=alpha_hat), offset=offset).fit(
        cov_type="cluster", cov_kwds={"groups": df["state"]})
    res = pd.DataFrame({"rate_ratio": np.exp(nb_total_full5.params).round(3), "p": nb_total_full5.pvalues.round(4)})
    save_csv("C3d", "rate_total_full5_predictor_fit", res)
    print("-- same 5-predictor set as rate_social's final model, refit for rate_total. This is NOT the selected "
          "model (that's the 1-predictor nb_total from C3c/C4) -- it exists purely so the coefficient comparison "
          "figure compares like-for-like architectures instead of a 5-predictor model against a 1-predictor one.")
    print(res)

run_step("C3d", _c3d)

### C4 -- Final model (selected predictor set)

In [ ]:
def _c4():
    global nb_total
    df = analysis_df.copy()
    offset = np.log(df["total_text_reviews"])
    nb_total = smf.glm(RATE_TOTAL_FORMULA, data=df, family=fam_total, offset=offset).fit(
        cov_type="cluster", cov_kwds={"groups": df["state"]})
    res = pd.DataFrame({"rate_ratio": np.exp(nb_total.params).round(3), "ci_low": np.exp(nb_total.conf_int()[0]).round(3),
                         "ci_high": np.exp(nb_total.conf_int()[1]).round(3), "p": nb_total.pvalues.round(4)})
    save_csv("C4", "final_model_rate_total", res)
    print(res)

run_step("C4", _c4)

### C5 -- Diagnostics: dispersion, Moran's I, VIF, Cook's distance

In [ ]:
def _c5():
    disp = nb_total.pearson_chi2 / nb_total.df_resid
    print(f"dispersion (Pearson chi2/df_resid) = {disp:.3f}")

    resid_df = analysis_df[["constituency_code"]].copy()
    resid_df["dev_resid"] = nb_total.resid_deviance
    merged = moran_gdf[["constituency_code"]].merge(resid_df, on="constituency_code", how="left")
    assert merged["constituency_code"].tolist() == moran_gdf["constituency_code"].tolist()
    moran = Moran(merged["dev_resid"].values, w, permutations=999)
    print(f"Moran's I (deviance resid) = {moran.I:.4f}, p = {moran.p_sim:.4f}")

    X = dmatrix(" + ".join(final_predictors_total), data=analysis_df, return_type="dataframe")
    vif = pd.DataFrame({"variable": X.columns, "vif": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]})
    vif = vif[vif.variable != "Intercept"]
    save_csv("C5", "vif", vif)
    print(vif.round(2).to_string(index=False))

    influence = nb_total.get_influence()
    cooks_d = influence.cooks_distance[0]
    top5 = analysis_df.assign(cooks_d=cooks_d).nlargest(5, "cooks_d")[["constituency_code", "constituency", "cooks_d"]]
    save_csv("C5", "top5_cooks_distance", top5)
    print(top5)

    RESULTS["C5_dispersion"] = round(float(disp), 3)
    RESULTS["C5_moran_I"] = round(float(moran.I), 4)
    RESULTS["C5_moran_p"] = round(float(moran.p_sim), 4)

run_step("C5", _c5)

### C6 -- Substantive results: predicted rates at representative predictor values

In [ ]:
def _c6():
    base_row = analysis_df[[p for p in candidate_predictors]].median().to_dict()
    base_row["is_east"] = 0
    base_row["in_split_city"] = 0
    base_row["total_text_reviews"] = analysis_df["total_text_reviews"].median()

    rows = []
    for p in [t.replace("_c", "") for t in final_predictors_total if t not in ("is_east", "in_split_city") and ":" not in t]:
        for pct, label in [(0.25, "p25"), (0.75, "p75")]:
            row = dict(base_row)
            row[p] = analysis_df[p].quantile(pct)
            rows.append({"predictor": p, "level": label, "value": row[p], **row})
    pred_df = pd.DataFrame(rows)
    for c in candidate_predictors:
        pred_df[c + "_c"] = pred_df[c] - analysis_df[c].mean()
    pred_df["offset"] = np.log(pred_df["total_text_reviews"])
    pred_df["predicted_count"] = nb_total.predict(pred_df, offset=pred_df["offset"])
    pred_df["predicted_rate_total"] = pred_df["predicted_count"] / pred_df["total_text_reviews"]

    out = pred_df[["predictor", "level", "value", "predicted_rate_total"]]
    save_csv("C6", "predicted_rates_at_p25_p75", out)
    print(out.round(5).to_string(index=False))
    print("\n-- predicted rate_total at the 25th vs 75th percentile of each surviving predictor, "
          "holding others at their sample median, gives effect sizes on a common (rate) scale "
          "comparable to the rate_social analysis.")

run_step("C6", _c6)

### C7 -- The bridge model: share_social (binomial)

In [ ]:
def _c7():
    global fit_share
    df = analysis_df.copy()
    formula_share = "social_reviews + non_social ~ " + " + ".join(RATE_SOCIAL_PREDICTORS)
    df["non_social"] = df["total_text_reviews"] - df["social_reviews"]
    fit_share = smf.glm(formula_share, data=df, family=sm.families.Binomial()).fit(cov_type="HC1")
    res = pd.DataFrame({"coef": fit_share.params, "ci_low": fit_share.conf_int()[0],
                         "ci_high": fit_share.conf_int()[1], "p": fit_share.pvalues})
    save_csv("C7", "bridge_model_share_social", res)
    print(res.round(4))

    resid_df = df[["constituency_code"]].copy()
    resid_df["dev_resid"] = fit_share.resid_deviance
    merged = moran_gdf[["constituency_code"]].merge(resid_df, on="constituency_code", how="left")
    assert merged["constituency_code"].tolist() == moran_gdf["constituency_code"].tolist()
    moran_share = Moran(merged["dev_resid"].values, w, permutations=999)
    print(f"Moran's I (share_social deviance resid) = {moran_share.I:.4f}, p = {moran_share.p_sim:.4f}")

    n_sig = int((fit_share.pvalues.drop("Intercept") < 0.05).sum())
    patterned = n_sig > 0
    print(f"\n{n_sig}/{len(RATE_SOCIAL_PREDICTORS)} predictors significant at p<0.05 -- "
          f"share_social is {'PATTERNED' if patterned else 'NOT patterned'} with respect to the predictors. "
          f"{'rate_total effects are therefore ambiguous between more racialising and more social reviewing.' if patterned else 'the two outcomes are near-equivalent; the choice between them is presentational.'}")
    RESULTS["C7_n_significant"] = n_sig

run_step("C7", _c7)

### C7b -- share_social refit as NB with log(total_text_reviews) offset

In [ ]:
def _c7b():
    global fit_share_nb
    df = analysis_df.copy()
    offset = np.log(df["total_text_reviews"])
    formula_share_nb = "social_reviews ~ " + " + ".join(RATE_SOCIAL_PREDICTORS)
    p0 = smf.glm(formula_share_nb, data=df, family=sm.families.Poisson(), offset=offset).fit()
    mu = p0.mu
    aux = sm.OLS(((df["social_reviews"] - mu) ** 2 - mu) / mu, mu).fit()
    alpha_hat = max(float(aux.params.iloc[0]), 1e-6)
    fit_share_nb = smf.glm(formula_share_nb, data=df, family=sm.families.NegativeBinomial(alpha=alpha_hat), offset=offset).fit(
        cov_type="cluster", cov_kwds={"groups": df["state"]})
    print(pd.DataFrame({"rate_ratio": np.exp(fit_share_nb.params).round(3), "p": fit_share_nb.pvalues.round(4)}))

run_step("C7b", _c7b)

### C7c -- Additivity check: beta_social + beta_share ~= beta_total

In [ ]:
def _c7c():
    rows = []
    for t in RATE_SOCIAL_PREDICTORS:
        b_social = nb_main.params[t]
        b_share = fit_share_nb.params[t]
        b_total = nb_total_full5.params[t]
        rows.append({"predictor": t, "beta_social": b_social, "beta_share": b_share,
                     "sum_social_plus_share": b_social + b_share, "beta_total_full5": b_total,
                     "diff": (b_social + b_share) - b_total})
    table = pd.DataFrame(rows)
    save_csv("C7c", "additivity_check", table)
    print(table.round(4).to_string(index=False))
    max_diff = table["diff"].abs().max()
    clean = max_diff < 0.05
    if clean:
        verdict = ("holds cleanly (the decomposition rate_total = rate_social x share_social is recovered by "
                    "the separately-fit models, so this is a mechanism, not just an assertion)")
    else:
        verdict = ("does NOT hold cleanly (the three separately-fit NB models do not recover the arithmetic "
                    "identity well -- treat the rate_social/share_social split as a descriptive decomposition, "
                    "not a verified mechanism)")
    print(f"\nmax |diff| = {max_diff:.4f} -- additivity {verdict}")
    RESULTS["C7c_additivity_max_diff"] = round(float(max_diff), 4)

run_step("C7c", _c7c)

### C8 -- Comparison deliverable (rate_social vs. rate_total vs. share_social)

In [ ]:
def _c8():
    all_terms = sorted(set(RATE_SOCIAL_PREDICTORS) | set(final_predictors_total))
    rows = []
    for t in all_terms:
        row = {"predictor": t}
        if t in nb_main.params.index:
            row["rate_social_coef"] = nb_main.params[t]
            ci = nb_main.conf_int()
            row["rate_social_ci_low"], row["rate_social_ci_high"] = ci[0][t], ci[1][t]
        if t in nb_total_full5.params.index:
            row["rate_total_coef_full5"] = nb_total_full5.params[t]
            ci = nb_total_full5.conf_int()
            row["rate_total_full5_ci_low"], row["rate_total_full5_ci_high"] = ci[0][t], ci[1][t]
        if t in nb_total.params.index:
            row["rate_total_coef_selected"] = nb_total.params[t]
            ci = nb_total.conf_int()
            row["rate_total_selected_ci_low"], row["rate_total_selected_ci_high"] = ci[0][t], ci[1][t]
        if "rate_social_coef" in row and "rate_total_coef_full5" in row:
            row["same_sign"] = np.sign(row["rate_social_coef"]) == np.sign(row["rate_total_coef_full5"])
            row["ci_overlap"] = not (row["rate_social_ci_high"] < row["rate_total_full5_ci_low"] or row["rate_total_full5_ci_high"] < row["rate_social_ci_low"])
        if t in fit_share.params.index:
            row["share_social_coef"] = fit_share.params[t]
        rows.append(row)
    table = pd.DataFrame(rows)
    save_csv("C8", "comparison_deliverable", table)
    print("-- rate_total_coef_full5 is the fair like-for-like comparison (same 5 predictors as rate_social); "
          "rate_total_coef_selected is from the model actually chosen by C3c's selection procedure (1 predictor).")
    print(table.round(4).to_string(index=False))

    model_level = pd.DataFrame({
        "model": ["rate_social (nb_main)", "rate_total, full 5-predictor (nb_total_full5)", "rate_total, selected (nb_total)", "share_social (fit_share, binomial)"],
        "AIC": [nb_main.aic, nb_total_full5.aic, nb_total.aic, fit_share.aic],
        "dispersion": [nb_main.pearson_chi2 / nb_main.df_resid, nb_total_full5.pearson_chi2 / nb_total_full5.df_resid,
                       nb_total.pearson_chi2 / nb_total.df_resid, fit_share.pearson_chi2 / fit_share.df_resid],
        "predictor_set": [str(RATE_SOCIAL_PREDICTORS), str(RATE_SOCIAL_PREDICTORS), str(final_predictors_total), str(RATE_SOCIAL_PREDICTORS)],
    })
    save_csv("C8", "model_level_comparison", model_level)
    print(model_level.round(3).to_string(index=False))

run_step("C8", _c8)